In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

C:\Users\CORE !5\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
from datetime import datetime, timedelta

In [3]:
from scipy import stats
from scipy.stats import pearsonr, spearmanr

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

In [4]:
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [5]:
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

In [6]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

In [7]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [8]:
from sklearn.inspection import permutation_importance

In [9]:
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

In [11]:
import os

# Main project directory
project_dir = r"C:\Users\CORE !5\Downloads\kampala-fuel-price-predictor"

# Folder containing the CSV datasets
data_dir = os.path.join(project_dir, "data", "raw")

print("Project Directory:", project_dir)
print("Data Directory:", data_dir)

Project Directory: C:\Users\CORE !5\Downloads\kampala-fuel-price-predictor
Data Directory: C:\Users\CORE !5\Downloads\kampala-fuel-price-predictor\data\raw


In [12]:
print("Files inside data/raw:")

for file in os.listdir(data_dir):
    print(file)

Files inside data/raw:
brent-weekly.csv
uganda-fuel-prices-ubos-2026-09.csv
uganda_fuel_taxes.csv
usd_ugx_exchange_rates.csv


In [14]:
# Configure paths to the four datasets

fuel_path = os.path.join(
    data_dir, "uganda-fuel-prices-ubos-2026-09.csv"
)

exchange_path = os.path.join(
    data_dir, "usd_ugx_exchange_rates.csv"
)

brent_path = os.path.join(
    data_dir, "brent-weekly.csv"
)

tax_path = os.path.join(
    data_dir, "uganda_fuel_taxes.csv"
)

print("All dataset paths configured!")

All dataset paths configured!


In [15]:
# Check whether each dataset exists

datasets = {
    "Fuel Prices": fuel_path,
    "Exchange Rates": exchange_path,
    "Brent Crude": brent_path,
    "Fuel Taxes": tax_path
}

for name, path in datasets.items():
    if os.path.exists(path):
        print(f"{name}: FOUND")
    else:
        print(f"{name}: NOT FOUND")

Fuel Prices: FOUND
Exchange Rates: FOUND
Brent Crude: FOUND
Fuel Taxes: FOUND


In [16]:
# Load datasets into Pandas DataFrames

fuel_df = pd.read_csv(fuel_path)

exchange_df = pd.read_csv(exchange_path)

brent_df = pd.read_csv(brent_path)

tax_df = pd.read_csv(tax_path)

print("All four datasets loaded successfully!")

All four datasets loaded successfully!


In [17]:
print("UGANDA FUEL PRICES")
display(fuel_df.head())

print("\nUSD/UGX EXCHANGE RATES")
display(exchange_df.head())

print("\nBRENT CRUDE PRICES")
display(brent_df.head())

print("\nUGANDA FUEL TAXES")
display(tax_df.head())

UGANDA FUEL PRICES


,month,petrol_shs_per_litre,diesel_shs_per_litre,paraffin_shs_per_litre,petrol_release,diesel_release,paraffin_release
0,2021-05,3971,3522,"3,098.00",2022-05,2022-05,2022-05
1,2021-06,4009,3516,"3,093.00",2022-06,2022-06,2022-06
2,2021-07,4020,3531,"3,101.00",2022-07,2022-07,2022-07
3,2021-08,4112,3661,"3,105.00",2022-08,2022-08,2022-08
4,2021-09,4190,3764,"3,105.00",2021-09,2021-09,2022-09



USD/UGX EXCHANGE RATES


,indicator,Unit,Date,Value
0,"Market Exchange Rate,Pd.End",NaN,2016M1,"3,479.55"
1,"Market Exchange Rate,Pd.End",NaN,2016M2,"3,348.55"
2,"Market Exchange Rate,Pd.End",NaN,2016M3,"3,374.43"
3,"Market Exchange Rate,Pd.End",NaN,2016M4,"3,325.55"
4,"Market Exchange Rate,Pd.End",NaN,2016M5,"3,364.52"



BRENT CRUDE PRICES


,Date,Price
0,1987-05-15,18.58
1,1987-05-22,18.54
2,1987-05-29,18.60
3,1987-06-05,18.70
4,1987-06-12,18.75



UGANDA FUEL TAXES


,effective_date,petrol_excise,diesel_excise
0,2024-07-01,1550,1230
1,2025-07-01,1550,1230
2,2026-07-01,1750,1430


In [18]:
dataset_summary = {
    "Fuel Prices": fuel_df.shape,
    "Exchange Rates": exchange_df.shape,
    "Brent Crude": brent_df.shape,
    "Fuel Taxes": tax_df.shape
}

for name, dimensions in dataset_summary.items():
    print(f"{name}: {dimensions[0]} rows, {dimensions[1]} columns")

Fuel Prices: 64 rows, 7 columns
Exchange Rates: 130 rows, 4 columns
Brent Crude: 2057 rows, 2 columns
Fuel Taxes: 3 rows, 3 columns


In [20]:
# Group datasets for easier inspection

all_datasets = {
    "Fuel Prices": fuel_df,
    "Exchange Rates": exchange_df,
    "Brent Crude": brent_df,
    "Fuel Taxes": tax_df
}

for name, df in all_datasets.items():

    print("\n" + "*" * 60)
    print(name.upper())
    print("*" * 60)

    print("\nDataset dimensions:")
    print(df.shape)

    print("\nColumn data types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    print("\nNumerical summary:")
    display(df.describe())


************************************************************
FUEL PRICES
************************************************************

Dataset dimensions:
(64, 7)

Column data types:
month                         str
petrol_shs_per_litre        int64
diesel_shs_per_litre        int64
paraffin_shs_per_litre    float64
petrol_release                str
diesel_release                str
paraffin_release              str
dtype: object

Missing values:
month                      0
petrol_shs_per_litre       0
diesel_shs_per_litre       0
paraffin_shs_per_litre    42
petrol_release             0
diesel_release             0
paraffin_release          42
dtype: int64

Duplicate rows:
0

Numerical summary:


,petrol_shs_per_litre,diesel_shs_per_litre,paraffin_shs_per_litre
count,64.00,64.00,22.00
mean,"5,271.92","4,950.69","3,924.09"
std,615.72,727.28,680.34
min,"3,971.00","3,516.00","3,093.00"
25%,"5,027.00","4,692.50","3,159.00"
50%,"5,138.50","4,850.50","4,287.50"
75%,"5,474.50","5,126.00","4,544.75"
max,"6,642.00","6,757.00","4,645.00"



************************************************************
EXCHANGE RATES
************************************************************

Dataset dimensions:
(130, 4)

Column data types:
indicator        str
Unit         float64
Date             str
Value        float64
dtype: object

Missing values:
indicator      0
Unit         130
Date           0
Value          0
dtype: int64

Duplicate rows:
0

Numerical summary:


,Unit,Value
count,0.00,130.00
mean,NaN,"3,669.21"
std,NaN,131.44
min,NaN,"3,325.55"
25%,NaN,"3,600.10"
50%,NaN,"3,684.50"
75%,NaN,"3,748.93"
max,NaN,"4,279.60"



************************************************************
BRENT CRUDE
************************************************************

Dataset dimensions:
(2057, 2)

Column data types:
Date         str
Price    float64
dtype: object

Missing values:
Date     0
Price    0
dtype: int64

Duplicate rows:
0

Numerical summary:


,Price
count,"2,057.00"
mean,51.86
std,33.10
min,9.44
25%,19.62
50%,47.15
75%,75.58
max,141.07



************************************************************
FUEL TAXES
************************************************************

Dataset dimensions:
(3, 3)

Column data types:
effective_date      str
petrol_excise     int64
diesel_excise     int64
dtype: object

Missing values:
effective_date    0
petrol_excise     0
diesel_excise     0
dtype: int64

Duplicate rows:
0

Numerical summary:


,petrol_excise,diesel_excise
count,3.00,3.00
mean,"1,616.67","1,296.67"
std,115.47,115.47
min,"1,550.00","1,230.00"
25%,"1,550.00","1,230.00"
50%,"1,550.00","1,230.00"
75%,"1,650.00","1,330.00"
max,"1,750.00","1,430.00"
